# Feature generation with an Autoencoder
The goal of this notebook is to generate features from the spectrograms of different mic channels.  
For this, an autoencoder will be used to encode the spectrograms into a low dimensional vector,  
which will later be passed to the classification model.

To produce enough training samples, spectrograms are generated from 2-second sliding windows across the recordings.

For this notebook we utilize a simpele train-test split where we leave out a car to validate the model.
For the final classifier the training of the autoencoder should adhere to the choosen split.
It might make sense to use a wrapper for the autoencoder later which inherits from `sklearn.base.BaseEstimator` and `sklearn.base.TransformerMixin`.

## Conditioning the latent vectors to be discriminative
To bias the latent representations to be better suited for later classification, we utilize a classifier that contributes to the loss.  
$\alpha$ controls the impact of the classifier.

In [1]:
# TODOS
# Instead of using the mel spectrogram, use the meal coefficients directly as features
# Apply some normalization
# Use 1D convolutions per mfcc feature over time
# Use Upconv in reconstruction

In [2]:
from pathlib import Path

import librosa
import lightning as pl
import numpy as np
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from tqdm import tqdm

from roar import ROOT_DIR
from roar.preprocessing import load_h5_channel

# Data Module
To keep all dataloading in one place, we will use a pytorch lighning datamodule

In [3]:
def generate_spectrograms(file: Path, channel_name: str) -> list[Tensor]:
    # Implementation for generating spectrograms from audio files
    data, sample_rate = load_h5_channel(file, channel_name)  # type: ignore
    data = data.astype(np.float32)

    window_duration = 2.0
    step_size = 0.1
    window_samples = int(window_duration * sample_rate)
    step_samples = int(step_size * sample_rate)

    if len(data) < window_samples:
        print(f"Audio too short ({len(data) / sample_rate:.2f}s < 2s), discarding: {file}")
        return []

    # High-pass filter (Tyre noise only happens between 400 and 5000 Hz)
    sos = scipy.signal.butter(4, 200, "hp", fs=sample_rate, output="sos")
    data = scipy.signal.sosfilt(sos, data)
    data = data / (np.max(np.abs(data)) + 1e-8)  # Normalize

    spectrograms = []
    for start_idx in range(0, len(data) - window_samples + 1, step_samples):
        window_data = data[start_idx : start_idx + window_samples]
        mel_spec = librosa.feature.mfcc(
            y=window_data, sr=sample_rate, n_mfcc=13, n_fft=2048, hop_length=512
        )

        spectrograms.append(mel_spec)

    tensor_spectrograms = [
        torch.tensor(spec).float() for spec in spectrograms
    ]  # Add channel dimension
    return tensor_spectrograms

In [4]:
class SpectrogramDataset(torch.utils.data.Dataset):
    """
    Dataset for loading spectrogram tensors with track labels.

    Folder structure:
        data_images → trackXXX → CarYYY → TyreZZZ → experiment_AAA → spec_000.pt

    Target: Track name (trackXXX)
    """

    def __init__(
        self,
        tensor_paths: list[Path],
        mean: torch.Tensor,
        std: torch.Tensor,
    ):
        """
        Args:
            tensor_paths: List of paths to .pt files
            mean: Mean tensor for normalization
            std: Std tensor for normalization
        """
        self.tensor_paths = tensor_paths
        self.mean = mean
        self.std = std

        # Extract track labels and create label mapping
        self.track_labels = self._extract_track_labels()
        self.unique_tracks = sorted(set(self.track_labels))
        self.track_to_idx = {track: idx for idx, track in enumerate(self.unique_tracks)}
        self.idx_to_track = {idx: track for track, idx in self.track_to_idx.items()}

        print(f"Dataset initialized with {len(self.tensor_paths)} samples")
        print(f"Number of unique tracks: {len(self.unique_tracks)}")
        print(f"Tracks: {self.unique_tracks}")
        print(f"Class distribution: {self._get_class_distribution()}")

    def _extract_track_labels(self) -> list[str]:
        """Extract track name from each file path."""
        labels = []
        for path in self.tensor_paths:
            # Path structure: .../trackXXX/CarYYY/TyreZZZ/experiment_AAA/spec_000.pt
            parts = path.parts
            # Find the track part (should be 5 levels up from the file)
            track_idx = (
                -5
            )  # spec_000.pt (-1), experiment_AAA (-2), TyreZZZ (-3), CarYYY (-4), trackXXX (-5)
            track = parts[track_idx]
            labels.append(track)
        return sorted(labels)

    def _get_class_distribution(self) -> dict[str, int]:
        """Get the distribution of samples per track."""
        distribution = {}
        for label in self.track_labels:
            distribution[label] = distribution.get(label, 0) + 1
        return distribution

    def normalize_spectrogram(self, spec: torch.Tensor) -> torch.Tensor:
        """
        Normalize spectrogram based on statistics type.

        Args:
            spec: Spectrogram tensor of shape (n_frequencies, time_frames)

        Returns:
            Normalized spectrogram
        """
        normalized = (spec - self.mean) / (self.std + 1e-8)
        return normalized

    def __len__(self) -> int:
        """Return the total number of samples."""
        return len(self.tensor_paths)

    def __getitem__(self, idx: int) -> tuple[Tensor, int]:
        """
        Get a single sample.

        Args:
            idx: Index of the sample

        Returns:
            Tuple of (normalized_spectrogram, track_label_idx)
        """
        # Load spectrogram
        spec_path = self.tensor_paths[idx]
        spec = torch.load(spec_path)

        # Normalize
        spec = self.normalize_spectrogram(spec)

        # Get label
        track_name = self.track_labels[idx]
        label = self.track_to_idx[track_name]

        return spec, label

    def get_class_weights(self) -> torch.Tensor:
        """
        Compute class weights for imbalanced datasets.
        Useful for weighted loss functions.

        Returns:
            Tensor of weights for each class
        """
        distribution = self._get_class_distribution()
        total_samples = len(self)
        num_classes = len(self.unique_tracks)

        weights = []
        for track in self.unique_tracks:
            count = distribution[track]
            weight = total_samples / (num_classes * count)
            weights.append(weight)

        return torch.tensor(weights, dtype=torch.float32)

In [5]:
class SpectralDataModule(pl.LightningDataModule):
    def __init__(
        self,
        source_path: Path,
        dest_path: Path = ROOT_DIR / "data_images",
        leave_out_car: str = "02_Q8 e-tron",
        batch_size: int = 32,
    ) -> None:
        super().__init__()
        self.source_path = source_path
        self.dest_path = dest_path
        self.leave_out_car = leave_out_car
        self.batch_size = batch_size
        self.channel_name = "Ch_1_labV12_cleaned"

    def prepare_data(self) -> None:
        """
        Generate the spectrograms for each audio file and save them to disk.
        """
        h5_files = list(self.source_path.glob("**/*.h5"))
        target_folders = []

        for h5_file in h5_files:
            parts = h5_file.relative_to(self.source_path).parts
            track = parts[0]  # trackXXX
            car = parts[1]  # CarYYY
            tyre = parts[2]  # TyreZZZ
            experiment = h5_file.stem  # experiment_AAA (without .h5)

            # Create destination folder
            output_dir = self.dest_path / track / car / tyre / experiment
            target_folders.append(output_dir)

        if not all(folder.exists() for folder in target_folders):
            for h5_file, output_dir in tqdm(zip(h5_files, target_folders)):
                if output_dir.exists():
                    continue  # Skip already processed files
                output_dir.mkdir(parents=True, exist_ok=True)
                spectrograms = generate_spectrograms(h5_file, channel_name=self.channel_name)
                # Save spectrograms to img_path or process as needed
                for i, spectrogram in enumerate(spectrograms):
                    save_file = output_dir / f"spec_{i:03d}.pt"
                    torch.save(spectrogram, save_file)

        # Calculate mean and std here
        all_tensors = []
        spectrogram_tensors = list(self.dest_path.glob("**/spec_*.pt"))
        train_tensors = [t for t in spectrogram_tensors if self.leave_out_car not in str(t)]
        for train_tensor in train_tensors:
            spectrogram = torch.load(train_tensor)
            all_tensors.append(spectrogram)
        full_data = torch.cat(all_tensors, dim=-1)

        self.mean = full_data.mean(dim=-1, keepdim=True)
        self.std = full_data.std(dim=-1, keepdim=True)

    def setup(self, stage: str | None = None) -> None:
        # For the fit stage, calc the mean and std of the dataset for normalization
        spectrogram_tensors = list(self.dest_path.glob("**/spec_*.pt"))
        if stage == "fit" or stage is None:
            train_tensors = [t for t in spectrogram_tensors if self.leave_out_car not in str(t)]
            val_tensors = [t for t in spectrogram_tensors if self.leave_out_car in str(t)]
            print(
                f"Found {len(train_tensors)} training spectrograms and {len(val_tensors)} validation spectrograms."
            )

            self.train_dataset = SpectrogramDataset(
                tensor_paths=train_tensors,
                mean=self.mean,
                std=self.std,
            )
            self.val_dataset = SpectrogramDataset(
                tensor_paths=val_tensors,
                mean=self.mean,
                std=self.std,
            )
        if stage == "validate":
            val_tensors = [t for t in spectrogram_tensors if self.leave_out_car in str(t)]
            print(f"Found {len(val_tensors)} validation spectrograms.")
            self.val_dataset = SpectrogramDataset(
                tensor_paths=val_tensors,
                mean=self.mean,
                std=self.std,
            )

    def train_dataloader(self) -> torch.utils.data.DataLoader:
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
        )

    def val_dataloader(self) -> torch.utils.data.DataLoader:
        return torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
        )

In [6]:
datamodule = SpectralDataModule(
    source_path=ROOT_DIR / "data_cleaned",
    dest_path=ROOT_DIR / "data_images",
    leave_out_car="02_Q8 e-tron",
    batch_size=8,
)

## Autoencoder

In [7]:
class ConvAutoencoderClassifier(pl.LightningModule):
    def __init__(
        self,
        latent_dim: int,
        num_classes: int,
        weight: Tensor,
        n_mfcc: int = 13,
        seq_len: int = 188,
        alpha: float = 1.0,
        lr: float = 1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.alpha = alpha
        self.lr = lr
        self.n_mfcc = n_mfcc
        self.seq_len = seq_len
        self.classifier_criterion = nn.CrossEntropyLoss(weight=weight)
        self.reconstruction_criterion = nn.MSELoss()

        # Encoder: 1D convolutions over time dimension
        # Input: [B, n_mfcc, seq_len] -> Output: [B, 128, seq_len//4]
        self.encoder = nn.Sequential(
            nn.Conv1d(n_mfcc, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.MaxPool1d(2),  # seq_len -> seq_len//2
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.MaxPool1d(2),  # seq_len//2 -> seq_len//4
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.SiLU(),
        )

        # Global pooling and latent space projection
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc_latent = nn.Linear(128, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 128)

        # Decoder: 1D transposed convolutions
        # Input: [B, 128, 1] -> Output: [B, n_mfcc, seq_len]
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=2, stride=2),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Conv1d(32, n_mfcc, kernel_size=3, padding=1),
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.SiLU(),
            nn.Linear(64, num_classes),
        )

    def encode(self, x: Tensor) -> Tensor:
        """Encode MFCC input to latent representation.

        Args:
            x: Input tensor of shape [B, n_mfcc, seq_len]

        Returns:
            Latent representation of shape [B, latent_dim]
        """
        x = self.encoder(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        z = self.fc_latent(x)
        return z

    def decode(self, z: Tensor, seq_len: int) -> Tensor:
        """Decode latent representation to MFCC reconstruction.

        Args:
            z: Latent tensor of shape [B, latent_dim]
            seq_len: Target sequence length (should be 188)

        Returns:
            Reconstructed MFCC of shape [B, n_mfcc, seq_len]
        """
        x = self.fc_decode(z)
        x = x.view(-1, 128, 1)

        # Calculate the size after decoder's transposed convolutions
        # Two stride-2 transposed convs will multiply length by 4
        decoder_output_len = seq_len // 4

        # Interpolate to the size that will result in correct output after decoder
        x = F.interpolate(x, size=decoder_output_len, mode="nearest")

        # Apply decoder
        x = self.decoder(x)

        # Final interpolation to ensure exact match (handles any rounding issues)
        if x.shape[-1] != seq_len:
            x = F.interpolate(x, size=seq_len, mode="linear", align_corners=False)

        return x

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        """Forward function of the autoencoder

        Args:
            x (Tensor): Input tensor representing MFCC features.
                       Assumed shape is (batch_size, n_mfcc, seq_len).

        Returns:
            tuple[Tensor, Tensor, Tensor]: Reconstructed output, classification logits,
                                          and latent representation.
        """
        seq_len = x.shape[-1]
        z = self.encode(x)
        recon = self.decode(z, seq_len)
        logits = self.classifier(z)
        return recon, logits, z

    def compute_loss(
        self,
        x: Tensor,
        recon: Tensor,
        logits: Tensor,
        y: Tensor,
    ) -> tuple[Tensor, Tensor, Tensor]:
        """Compute total loss combining reconstruction MSE and classification cross-entropy.

        Args:
            x (Tensor): Input MFCC features of shape (batch_size, n_mfcc, seq_len).
            recon (Tensor): Reconstructed MFCC features matching the shape of `x`.
            logits (Tensor): Classifier logits of shape (batch_size, num_classes).
            y (Tensor): Ground-truth class labels of shape (batch_size,).

        Returns:
            tuple[Tensor, Tensor, Tensor]: Total loss, reconstruction loss, and classification loss.
        """
        recon_loss = self.reconstruction_criterion(recon, x)
        clf_loss = self.classifier_criterion(logits, y)
        total_loss = recon_loss + self.alpha * clf_loss
        return total_loss, recon_loss, clf_loss

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> Tensor:
        x, y = batch
        recon, logits, _ = self.forward(x)
        total_loss, recon_loss, clf_loss = self.compute_loss(x, recon, logits, y)

        self.log("train/total_loss", total_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train/recon_loss", recon_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train/clf_loss", clf_loss, on_step=False, on_epoch=True, prog_bar=True)

        return total_loss

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> None:
        x, y = batch
        recon, logits, _ = self.forward(x)
        total_loss, recon_loss, clf_loss = self.compute_loss(x, recon, logits, y)

        self.log("val/total_loss", total_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/recon_loss", recon_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/clf_loss", clf_loss, on_step=False, on_epoch=True, prog_bar=True)

    def predict_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int, dataloader_idx: int = 0
    ) -> tuple[Tensor, Tensor]:
        x, _ = batch
        _, logits, z = self.forward(x)
        return torch.argmax(logits, dim=1), z

    def configure_optimizers(self):  # type: ignore
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr)
        lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.3, patience=7
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": lr_scheduler,
                "monitor": "val/total_loss",
                "step": "epoch",
            },
        }

## Training and Logging

In [8]:
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

In [10]:
datamodule.prepare_data()
datamodule.setup(stage="fit")

model = ConvAutoencoderClassifier(
    latent_dim=128,
    num_classes=2,
    alpha=1.0,
    lr=3e-4,
    weight=datamodule.train_dataset.get_class_weights(),
)

Found 4537 training spectrograms and 2296 validation spectrograms.
Dataset initialized with 4537 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 1189, 'track211': 3348}
Dataset initialized with 2296 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 421, 'track211': 1875}


In [ ]:
logger = TensorBoardLogger(ROOT_DIR / "tb_logs", name="autoencoder_classifier")
callbacks = [
    EarlyStopping(monitor="val/total_loss", patience=20, mode="min"),
    LearningRateMonitor(logging_interval="epoch"),
]
trainer = pl.Trainer(logger=logger, max_epochs=300, callbacks=callbacks)  # type: ignore
model.compile()
trainer.fit(model, datamodule=datamodule)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/moritzfeik/Developer/ROAR/.pixi/envs/default/lib/python3.13/site-packages/lightning/pytorch/core/optimizer.py:259: Found unsupported keys in the lr scheduler dict: {'step'}. HINT: remove them from the output of `configure_optimizers`.

  | Name                     | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------------------
0 | classifier_criterion     | CrossEntropyLoss  | 0      | train | 0    
1 | reconstruction_criterion | MSELoss           | 0      | train | 0    
2 | encoder                  | Sequential        | 32.6 K | train | 0    
3 | global_pool              | AdaptiveAvgPool1d | 0      | train | 0    
4 | fc_latent          

Found 4537 training spectrograms and 2296 validation spectrograms.
Dataset initialized with 4537 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 1189, 'track211': 3348}
Dataset initialized with 2296 samples
Number of unique tracks: 2
Tracks: ['track150', 'track211']
Class distribution: {'track150': 421, 'track211': 1875}
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/moritzfeik/Developer/ROAR/.pixi/envs/default/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


/Users/moritzfeik/Developer/ROAR/.pixi/envs/default/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 5:  95%|█████████▌| 540/568 [00:10<00:00, 53.42it/s, v_num=1, val/total_loss=1.560, val/recon_loss=0.687, val/clf_loss=0.874, train/total_loss=0.731, train/recon_loss=0.642, train/clf_loss=0.0897]